In [35]:
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command
from langgraph.graph.message import MessagesState
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from typing import NotRequired

load_dotenv()


True

In [36]:
class AgentsState(MessagesState):
    current_agent: NotRequired[str]
    transfered_by: NotRequired[str]


llm = init_chat_model("openai:gpt-4o")

In [37]:
def make_agent(prompt, tools):
    def agent_node(state: AgentsState):
        llm_with_tools = llm.bind_tools(tools)
        response = llm_with_tools.invoke(
            f"""
        {prompt}

        Conversation history:
        {state["messages"]}
        """
        )
        return {"messages": [response]}

    agent_builder = StateGraph(AgentsState)

    agent_builder.add_node("agent", agent_node)

    agent_builder.add_node("tools", ToolNode(tools))

    agent_builder.add_edge(START, "agent")

    agent_builder.add_conditional_edges("agent", tools_condition)
    agent_builder.add_edge("tools", "agent")
    agent_builder.add_edge("agent", END)

    return agent_builder.compile()

In [38]:
@tool
def handoff_tool(transfer_to: str, transfered_by: str):
    """
    Handoff the another agent.

    Use this tool when the customer speaks a language that you don't understate.


    Possisble values for `transfer_to`:

    - `korean_agent`
    - `greek_agent`
    - `spanish_agent`

    Possisble values for `transfered_by`:
    - `korean_agent`
    - `greek_agent`
    - `spanish_agent`

    Args:
    - transfer_to (str): The agent to transfer the conversation to.
    - transfered_by (str): The agent who is transferring the conversation.
    """


    return Command(
        update={
            "current_agent": transfer_to,
            "transfered_by": transfered_by,
        },
        goto=transfer_to,
        graph=Command.PARENT
    )

In [ ]:
graph_builder = StateGraph(AgentsState)

graph_builder.add_node(
    "korean_agent",
    make_agent(
        prompt="You are a Korean customer support agent, You only speak and understand Korean.",
        tools=[handoff_tool],
    ),
    destinations=("greek_agent", "spanish_agent")
)

graph_builder.add_node(
    "greek_agent",
    make_agent(
        prompt="You are a Greek customer support agent, You only speak and understand Greek.",
        tools=[handoff_tool],
    ),
    destinations=("korean_agent", "spanish_agent"),
)

graph_builder.add_node(
    "spanish_agent",
    make_agent(
        prompt="You are a Spanish customer support agent, You only speak and understand Spanish.",
        tools=[handoff_tool],
    ),
    destinations=("greek_agent", "spanish_agent"),
)

graph_builder.add_edge(START, "korean_agent")

graph = graph_builder.compile()

In [40]:
input_state: AgentsState = {
    "messages": [HumanMessage(content="Hola! Necesito ayuda con mi cuenta")]
}

for event in graph.stream(input_state):
    print(event)

{'korean_agent': {'current_agent': 'spanish_agent', 'transfered_by': 'korean_agent'}}
{'spanish_agent': {'messages': [HumanMessage(content='Hola! Necesito ayuda con mi cuenta', additional_kwargs={}, response_metadata={}, id='9a84aa9b-2f05-421a-81c5-72d4701a2220'), AIMessage(content='¡Hola! ¿En qué puedo ayudarte con tu cuenta?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 13, 'prompt_tokens': 232, 'total_tokens': 245, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_ae736abb0d', 'id': 'chatcmpl-DcJjFK3P8yFZWNoERt7b369AO1xcL', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--019dfa87-6cf2-7542-b06f-ea377cee490e-0', usage_metadata={'input_tokens': 232, 'output_tokens': 13, 'total_tokens': 245, '